# 17 — Assembly101 Visual-Only LTContext Baselines

This notebook trains two controlled visual-only LTContext baselines:

```text
1. LTContext + ProcedureVRL hidden video features [512, 16]
2. LTContext + CLIP ViT-B/16 visual features       [512, 16]
```

The implementation follows the main architecture of the official LTContext
model:

- dilated temporal convolutions;
- local windowed self-attention;
- sparse long-term attention across strided temporal positions;
- four-stage prediction refinement;
- previous-stage features used as attention values in refinement stages;
- cross-entropy plus temporal smoothing loss.

The original Assembly101 LTContext configuration was designed for much longer
feature sequences and uses attention groups/windows of 64. Our extracted
representations contain only 16 temporal positions, so this notebook uses a
documented compact adaptation:

```text
local attention radius = 2 positions
long-term attention stride = 4 positions
maximum effective convolution dilation = 8
```

This is a **controlled LTContext adaptation for the current `[512, 16]`
feature representation**, not an exact reproduction of the official
Assembly101 TSM-feature experiment.

Both feature experiments use:

- the same official Assembly101 train/validation/test split;
- identical LTContext architecture and optimization settings;
- train-only feature normalization;
- validation-only checkpoint selection;
- one final test evaluation after model selection;
- video features only at training and inference.

The notebook also loads the completed MS-TCN results from notebook 16 and
writes a combined comparison table.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Imports

In [2]:
from pathlib import Path
from datetime import datetime, timezone

import gc
import json
import math
import os
import random
import shutil
import time
import traceback

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

print("Python device information")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU",
)

Python device information
PyTorch: 2.11.0+cpu
CUDA available: False
GPU: CPU


## 3. Paths and experiment configuration

In [3]:
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/mmf_tas_lab_data"
)

FEATURE_EXTRACTION_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "assembly101"
    / "coarse_mstcn_format"
    / "streaming_visual_features_v1"
)

PROCEDUREVRL_DATA_ROOT = (
    FEATURE_EXTRACTION_ROOT
    / "procedurevrl_hidden"
)
CLIP_DATA_ROOT = (
    FEATURE_EXTRACTION_ROOT
    / "clip_vitb16"
)

FEATURE_EXTRACTION_SUMMARY = (
    FEATURE_EXTRACTION_ROOT
    / "streaming_feature_extraction_summary.json"
)
AUTOLOOP_SUMMARY = (
    FEATURE_EXTRACTION_ROOT
    / "autoloop_latest_session_summary.json"
)

MS_TCN_RESULTS_ROOT = (
    FEATURE_EXTRACTION_ROOT
    / "runs"
    / "16_visual_only_mstcn"
)

OUT_ROOT = (
    FEATURE_EXTRACTION_ROOT
    / "runs"
    / "17_visual_only_ltcontext"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Run configuration
# ------------------------------------------------------------------
RUN_MODE = "full"
# RUN_MODE = "smoke"

SEED = 7

RUN_CONFIGS = {
    "smoke": {
        "epochs": 3,
        "max_train_sequences": 64,
        "max_val_sequences": 32,
        "max_test_sequences": 32,
        "eval_every": 1,
        "patience_evals": None,
        "warmup_epochs": 1,
    },
    "full": {
        "epochs": 120,
        "max_train_sequences": None,
        "max_val_sequences": None,
        "max_test_sequences": None,
        "eval_every": 5,
        "patience_evals": 12,
        "warmup_epochs": 15,
    },
}

if RUN_MODE not in RUN_CONFIGS:
    raise ValueError(
        f"Unknown RUN_MODE: {RUN_MODE}"
    )

RUN_CFG = RUN_CONFIGS[RUN_MODE]

# ------------------------------------------------------------------
# Official-inspired Assembly101 optimization settings
# ------------------------------------------------------------------
BATCH_SIZE = 64
NUM_WORKERS = 0

LEARNING_RATE = 2.5e-4
WEIGHT_DECAY = 1e-3
GRAD_CLIP = 5.0

# ------------------------------------------------------------------
# LTContext architecture
# ------------------------------------------------------------------
NUM_STAGES = 4
NUM_LAYERS = 9
MODEL_DIM = 64
DIM_REDUCTION = 2
REFINEMENT_DIM = MODEL_DIM // DIM_REDUCTION

NUM_ATTENTION_HEADS = 4
ATTENTION_DROPOUT = 0.1
BLOCK_DROPOUT = 0.5
CHANNEL_MASKING_PROB = 0.3

DILATION_FACTOR = 2
MAX_EFFECTIVE_DILATION = 8

# Compact adaptation for temporal length T=16.
LOCAL_ATTENTION_RADIUS = 2
LONG_TERM_ATTENTION_STRIDE = 4

SMOOTHING_LOSS_WEIGHT = 0.17
SMOOTHING_CLIP_VALUE = 16.0

STANDARDIZE_FEATURES = True
EXPECTED_FEATURE_DIM = 512
EXPECTED_TEMPORAL_LENGTH = 16
EXPECTED_NUM_CLASSES = 202

RESUME_IF_AVAILABLE = True
FORCE_RETRAIN_COMPLETED = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("RUN_MODE:", RUN_MODE)
print("RUN_CFG:", RUN_CFG)
print("DEVICE:", DEVICE)
print("OUT_ROOT:", OUT_ROOT)
print("LTContext stages/layers:", NUM_STAGES, NUM_LAYERS)
print("Model/refinement dimensions:", MODEL_DIM, REFINEMENT_DIM)
print("Local attention radius:", LOCAL_ATTENTION_RADIUS)
print("Long-term attention stride:", LONG_TERM_ATTENTION_STRIDE)

RUN_MODE: full
RUN_CFG: {'epochs': 120, 'max_train_sequences': None, 'max_val_sequences': None, 'max_test_sequences': None, 'eval_every': 5, 'patience_evals': 12, 'warmup_epochs': 15}
DEVICE: cpu
OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext
LTContext stages/layers: 4 9
Model/refinement dimensions: 64 32
Local attention radius: 2
Long-term attention stride: 4


## 4. Verify that feature extraction is complete

In [4]:
required_paths = {
    "feature extraction root": FEATURE_EXTRACTION_ROOT,
    "ProcedureVRL dataset": PROCEDUREVRL_DATA_ROOT,
    "CLIP dataset": CLIP_DATA_ROOT,
    "feature extraction summary": FEATURE_EXTRACTION_SUMMARY,
    "AUTOLOOP summary": AUTOLOOP_SUMMARY,
}

for name, path in required_paths.items():
    print(f"{name}: {path} -> {path.exists()}")
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required path: {name}: {path}"
        )

feature_summary = json.loads(
    FEATURE_EXTRACTION_SUMMARY.read_text(
        encoding="utf-8"
    )
)
autoloop_summary = json.loads(
    AUTOLOOP_SUMMARY.read_text(
        encoding="utf-8"
    )
)

progress = autoloop_summary.get(
    "final_progress",
    {}
)

print("Feature extraction summary status:", feature_summary.get("status"))
print("AUTOLOOP status:", autoloop_summary.get("status"))
print("AUTOLOOP progress:")
print(json.dumps(progress, indent=2))

assert progress.get("complete_recordings") == 350
assert progress.get("total_recordings") == 350
assert progress.get("complete_sequences") == 680
assert progress.get("total_sequences") == 680
assert progress.get("all_complete") is True

print("Assembly101 feature extraction: COMPLETE")

feature extraction root: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1 -> True
ProcedureVRL dataset: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/procedurevrl_hidden -> True
CLIP dataset: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16 -> True
feature extraction summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/streaming_feature_extraction_summary.json -> True
AUTOLOOP summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/autoloop_latest_session_summary.json -> True
Feature extraction summary status: completed_all_features
AUTOLOOP status: completed_all_features
AUTOLOOP progress:
{
  "complete_recordings": 350

## 5. Reproducibility

In [5]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Convolution results are normally deterministic for this model.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(SEED)
print("Seed:", SEED)

Seed: 7


## 6. Resolve mapping and official split bundles

In [6]:
def read_nonempty_lines(path):
    return [
        line.strip()
        for line in Path(path).read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip()
    ]


def resolve_bundle(split_dir, split_name):
    split_dir = Path(split_dir)

    exact_candidates = [
        split_dir / f"{split_name}.bundle",
        split_dir / f"{split_name}.split1.bundle",
    ]

    for candidate in exact_candidates:
        if candidate.exists():
            return candidate

    candidates = sorted(
        path
        for path in split_dir.glob("*.bundle")
        if split_name.lower() in path.name.lower()
        and ".partial." not in path.name.lower()
        and not path.name.lower().endswith(".partial.bundle")
    )

    if len(candidates) == 1:
        return candidates[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve {split_name} bundle "
        f"in {split_dir}. Candidates: {candidates}"
    )


def read_bundle(path):
    ids = []

    for line in read_nonempty_lines(path):
        # Bundle entries are normally <sequence_id>.txt.
        sequence_id = Path(line).stem
        ids.append(sequence_id)

    if len(ids) != len(set(ids)):
        raise ValueError(
            f"Duplicate sequence IDs in {path}"
        )

    return ids


def load_mapping(path):
    id_to_label = {}
    label_to_id = {}

    for line in read_nonempty_lines(path):
        index_text, label = line.split(
            maxsplit=1
        )
        index = int(index_text)

        if index in id_to_label:
            raise ValueError(
                f"Duplicate class ID {index}"
            )
        if label in label_to_id:
            raise ValueError(
                f"Duplicate label {label}"
            )

        id_to_label[index] = label
        label_to_id[label] = index

    return id_to_label, label_to_id


MAPPING_PATH = PROCEDUREVRL_DATA_ROOT / "mapping.txt"
SPLIT_DIR = PROCEDUREVRL_DATA_ROOT / "splits"

TRAIN_BUNDLE = resolve_bundle(
    SPLIT_DIR,
    "train",
)
VAL_BUNDLE = resolve_bundle(
    SPLIT_DIR,
    "val",
)
TEST_BUNDLE = resolve_bundle(
    SPLIT_DIR,
    "test",
)

train_ids_full = read_bundle(TRAIN_BUNDLE)
val_ids_full = read_bundle(VAL_BUNDLE)
test_ids_full = read_bundle(TEST_BUNDLE)

id_to_label, label_to_id = load_mapping(
    MAPPING_PATH
)
num_classes = len(id_to_label)

assert set(id_to_label) == set(
    range(num_classes)
)
assert num_classes == EXPECTED_NUM_CLASSES

train_set = set(train_ids_full)
val_set = set(val_ids_full)
test_set = set(test_ids_full)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)
assert len(train_set | val_set | test_set) == 680

print("Mapping:", MAPPING_PATH)
print("Classes:", num_classes)
print("Train bundle:", TRAIN_BUNDLE, len(train_ids_full))
print("Validation bundle:", VAL_BUNDLE, len(val_ids_full))
print("Test bundle:", TEST_BUNDLE, len(test_ids_full))
print("Total unique sequences:", len(train_set | val_set | test_set))

Mapping: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/procedurevrl_hidden/mapping.txt
Classes: 202
Train bundle: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/procedurevrl_hidden/splits/train.split1.bundle 393
Validation bundle: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/procedurevrl_hidden/splits/val.split1.bundle 120
Test bundle: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/procedurevrl_hidden/splits/test.split1.bundle 167
Total unique sequences: 680


## 7. Apply smoke/full subset configuration

In [7]:
def limit_ids(ids, maximum):
    if maximum is None:
        return list(ids)
    return list(ids)[: int(maximum)]


train_ids = limit_ids(
    train_ids_full,
    RUN_CFG["max_train_sequences"],
)
val_ids = limit_ids(
    val_ids_full,
    RUN_CFG["max_val_sequences"],
)
test_ids = limit_ids(
    test_ids_full,
    RUN_CFG["max_test_sequences"],
)

print("Active train sequences:", len(train_ids))
print("Active validation sequences:", len(val_ids))
print("Active test sequences:", len(test_ids))

assert train_ids
assert val_ids
assert test_ids

Active train sequences: 393
Active validation sequences: 120
Active test sequences: 167


## 8. Validate both LTContext input datasets

In [8]:
REPRESENTATIONS = {
    "procedurevrl_hidden": {
        "display_name": "ProcedureVRL hidden",
        "data_root": PROCEDUREVRL_DATA_ROOT,
        "feature_description": (
            "ProcedureVRL model.head hidden video embeddings"
        ),
    },
    "clip_vitb16": {
        "display_name": "CLIP ViT-B/16",
        "data_root": CLIP_DATA_ROOT,
        "feature_description": (
            "CLIP ViT-B/16 normalized visual frame embeddings"
        ),
    },
}

all_active_ids = (
    list(train_ids)
    + list(val_ids)
    + list(test_ids)
)

integrity_rows = []

for representation, config in REPRESENTATIONS.items():
    data_root = Path(config["data_root"])
    feature_dir = data_root / "features"
    gt_dir = data_root / "groundTruth"
    mapping_path = data_root / "mapping.txt"

    if not feature_dir.exists():
        raise FileNotFoundError(feature_dir)
    if not gt_dir.exists():
        raise FileNotFoundError(gt_dir)
    if not mapping_path.exists():
        raise FileNotFoundError(mapping_path)

    rep_id_to_label, rep_label_to_id = load_mapping(
        mapping_path
    )

    if rep_id_to_label != id_to_label:
        raise ValueError(
            f"Mapping differs for {representation}"
        )

    for sequence_id in tqdm(
        all_active_ids,
        desc=f"Validate {representation}",
    ):
        feature_path = (
            feature_dir
            / f"{sequence_id}.npy"
        )
        gt_path = (
            gt_dir
            / f"{sequence_id}.txt"
        )

        if not feature_path.exists():
            raise FileNotFoundError(
                feature_path
            )
        if not gt_path.exists():
            raise FileNotFoundError(gt_path)

        feature = np.load(
            feature_path,
            mmap_mode="r",
        )
        labels = read_nonempty_lines(gt_path)

        unknown_labels = sorted(
            set(labels)
            - set(label_to_id)
        )

        finite = bool(
            np.isfinite(feature).all()
        )

        row = {
            "representation": representation,
            "sequence_id": sequence_id,
            "split": (
                "train"
                if sequence_id in set(train_ids)
                else (
                    "val"
                    if sequence_id in set(val_ids)
                    else "test"
                )
            ),
            "feature_dim": int(
                feature.shape[0]
            ) if feature.ndim == 2 else -1,
            "feature_len": int(
                feature.shape[1]
            ) if feature.ndim == 2 else -1,
            "gt_len": len(labels),
            "finite": finite,
            "unknown_label_count": len(
                unknown_labels
            ),
        }
        integrity_rows.append(row)

        if feature.shape != (
            EXPECTED_FEATURE_DIM,
            EXPECTED_TEMPORAL_LENGTH,
        ):
            raise ValueError(
                f"Unexpected feature shape for "
                f"{representation}/{sequence_id}: "
                f"{feature.shape}"
            )

        if len(labels) != EXPECTED_TEMPORAL_LENGTH:
            raise ValueError(
                f"Unexpected GT length for "
                f"{representation}/{sequence_id}: "
                f"{len(labels)}"
            )

        if not finite:
            raise ValueError(
                f"NaN/Inf in {feature_path}"
            )

        if unknown_labels:
            raise ValueError(
                f"Unknown labels in {gt_path}: "
                f"{unknown_labels}"
            )

integrity_df = pd.DataFrame(
    integrity_rows
)
integrity_path = (
    OUT_ROOT
    / f"dataset_integrity_{RUN_MODE}.csv"
)
integrity_df.to_csv(
    integrity_path,
    index=False,
)

display(
    integrity_df.groupby(
        ["representation", "split"]
    ).agg(
        sequences=("sequence_id", "size"),
        feature_dims=("feature_dim", "nunique"),
        feature_lengths=("feature_len", "nunique"),
        gt_lengths=("gt_len", "nunique"),
        finite=("finite", "all"),
        unknown_labels=(
            "unknown_label_count",
            "sum",
        ),
    ).reset_index()
)

print("Saved:", integrity_path)

Validate procedurevrl_hidden:   0%|          | 0/680 [00:00<?, ?it/s]

Validate clip_vitb16:   0%|          | 0/680 [00:00<?, ?it/s]

,representation,split,sequences,feature_dims,feature_lengths,gt_lengths,finite,unknown_labels
0,clip_vitb16,test,167,1,1,1,True,0
1,clip_vitb16,train,393,1,1,1,True,0
2,clip_vitb16,val,120,1,1,1,True,0
3,procedurevrl_hidden,test,167,1,1,1,True,0
4,procedurevrl_hidden,train,393,1,1,1,True,0
5,procedurevrl_hidden,val,120,1,1,1,True,0


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/dataset_integrity_full.csv


## 9. Report class coverage by split

In [9]:
def labels_for_ids(data_root, sequence_ids):
    gt_dir = Path(data_root) / "groundTruth"
    labels = []

    for sequence_id in sequence_ids:
        labels.extend(
            read_nonempty_lines(
                gt_dir
                / f"{sequence_id}.txt"
            )
        )

    return labels


coverage_rows = []

for split_name, split_ids in [
    ("train", train_ids),
    ("val", val_ids),
    ("test", test_ids),
]:
    labels = labels_for_ids(
        PROCEDUREVRL_DATA_ROOT,
        split_ids,
    )
    present = set(labels)

    coverage_rows.append({
        "split": split_name,
        "num_sequences": len(split_ids),
        "num_temporal_positions": len(labels),
        "num_classes_present": len(present),
        "num_classes_missing": (
            num_classes - len(present)
        ),
    })

coverage_df = pd.DataFrame(
    coverage_rows
)
coverage_path = (
    OUT_ROOT
    / f"class_coverage_{RUN_MODE}.csv"
)
coverage_df.to_csv(
    coverage_path,
    index=False,
)

display(coverage_df)
print("Saved:", coverage_path)

,split,num_sequences,num_temporal_positions,num_classes_present,num_classes_missing
0,train,393,6288,188,14
1,val,120,1920,154,48
2,test,167,2672,164,38


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/class_coverage_full.csv


## 10. Feature normalization and dataset

In [10]:
def compute_train_stats(
    feature_dir,
    sequence_ids,
):
    feature_dir = Path(feature_dir)

    total_sum = None
    total_sumsq = None
    total_count = 0

    for sequence_id in tqdm(
        sequence_ids,
        desc="Train feature statistics",
    ):
        feature = np.load(
            feature_dir
            / f"{sequence_id}.npy"
        ).astype(np.float64)

        if total_sum is None:
            total_sum = np.zeros(
                feature.shape[0],
                dtype=np.float64,
            )
            total_sumsq = np.zeros(
                feature.shape[0],
                dtype=np.float64,
            )

        total_sum += feature.sum(axis=1)
        total_sumsq += (
            feature ** 2
        ).sum(axis=1)
        total_count += feature.shape[1]

    mean = total_sum / total_count
    variance = (
        total_sumsq / total_count
        - mean ** 2
    )
    variance = np.maximum(
        variance,
        1e-12,
    )
    std = np.sqrt(variance)

    return (
        mean.astype(np.float32),
        std.astype(np.float32),
    )


class Assembly101FeatureDataset(Dataset):
    def __init__(
        self,
        sequence_ids,
        data_root,
        label_to_id,
        mean=None,
        std=None,
    ):
        self.sequence_ids = list(
            sequence_ids
        )
        self.data_root = Path(
            data_root
        )
        self.feature_dir = (
            self.data_root / "features"
        )
        self.gt_dir = (
            self.data_root / "groundTruth"
        )
        self.label_to_id = dict(
            label_to_id
        )
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.sequence_ids)

    def __getitem__(self, index):
        sequence_id = self.sequence_ids[
            index
        ]

        feature = np.load(
            self.feature_dir
            / f"{sequence_id}.npy"
        ).astype(np.float32)

        labels = read_nonempty_lines(
            self.gt_dir
            / f"{sequence_id}.txt"
        )
        targets = np.asarray(
            [
                self.label_to_id[label]
                for label in labels
            ],
            dtype=np.int64,
        )

        if (
            self.mean is not None
            and self.std is not None
        ):
            feature = (
                feature
                - self.mean[:, None]
            ) / (
                self.std[:, None]
                + 1e-8
            )

        return {
            "sequence_id": sequence_id,
            "features": torch.from_numpy(
                feature
            ),
            "labels": torch.from_numpy(
                targets
            ),
        }


print("Dataset classes ready.")

Dataset classes ready.


## 11. LTContext model adapted to temporal length 16

In [11]:
class LocalWindowAttention(nn.Module):
    """Local self/cross attention in a fixed temporal neighborhood."""

    def __init__(
        self,
        model_dim,
        num_heads,
        radius,
        dropout,
    ):
        super().__init__()

        self.radius = int(radius)
        self.attention = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

    def forward(
        self,
        query_key,
        value_context=None,
    ):
        # Input format: [batch, channels, time]
        query = query_key.transpose(1, 2)
        value = (
            query
            if value_context is None
            else value_context.transpose(1, 2)
        )

        time_steps = query.shape[1]
        positions = torch.arange(
            time_steps,
            device=query.device,
        )

        distance = torch.abs(
            positions[:, None]
            - positions[None, :]
        )

        # True entries are forbidden in PyTorch attention masks.
        attention_mask = (
            distance > self.radius
        )

        output, _ = self.attention(
            query=query,
            key=query,
            value=value,
            attn_mask=attention_mask,
            need_weights=False,
        )

        return output.transpose(1, 2)


class StridedLongTermAttention(nn.Module):
    """Sparse full-context attention over positions sharing a stride offset.

    For stride=4 and T=16, attention groups are:

        [0, 4, 8, 12]
        [1, 5, 9, 13]
        [2, 6, 10, 14]
        [3, 7, 11, 15]

    Each group spans the complete sequence while avoiding dense T×T attention.
    """

    def __init__(
        self,
        model_dim,
        num_heads,
        stride,
        dropout,
    ):
        super().__init__()

        self.stride = int(stride)
        self.attention = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

    def forward(
        self,
        query_key,
        value_context=None,
    ):
        query_all = query_key.transpose(
            1,
            2,
        )
        value_all = (
            query_all
            if value_context is None
            else value_context.transpose(
                1,
                2,
            )
        )

        output_all = torch.zeros_like(
            query_all
        )
        time_steps = query_all.shape[1]

        for offset in range(
            min(
                self.stride,
                time_steps,
            )
        ):
            indices = torch.arange(
                offset,
                time_steps,
                self.stride,
                device=query_all.device,
            )

            if indices.numel() == 0:
                continue

            query = query_all.index_select(
                1,
                indices,
            )
            value = value_all.index_select(
                1,
                indices,
            )

            output, _ = self.attention(
                query=query,
                key=query,
                value=value,
                need_weights=False,
            )

            output_all.index_copy_(
                1,
                indices,
                output,
            )

        return output_all.transpose(1, 2)


class DilatedTemporalConv(nn.Module):
    def __init__(
        self,
        channels,
        dilation,
    ):
        super().__init__()

        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
        )

    def forward(self, x):
        return F.gelu(
            self.conv(x)
        )


class LTContextBlock(nn.Module):
    def __init__(
        self,
        model_dim,
        dilation,
        num_heads,
        local_radius,
        long_term_stride,
        attention_dropout,
        block_dropout,
    ):
        super().__init__()

        self.dilated_conv = (
            DilatedTemporalConv(
                channels=model_dim,
                dilation=dilation,
            )
        )

        self.norm_local = nn.InstanceNorm1d(
            model_dim,
            affine=True,
        )
        self.local_attention = (
            LocalWindowAttention(
                model_dim=model_dim,
                num_heads=num_heads,
                radius=local_radius,
                dropout=attention_dropout,
            )
        )

        self.norm_long_term = (
            nn.InstanceNorm1d(
                model_dim,
                affine=True,
            )
        )
        self.long_term_attention = (
            StridedLongTermAttention(
                model_dim=model_dim,
                num_heads=num_heads,
                stride=long_term_stride,
                dropout=attention_dropout,
            )
        )

        self.output_projection = nn.Conv1d(
            model_dim,
            model_dim,
            kernel_size=1,
        )
        self.dropout = nn.Dropout(
            block_dropout
        )

    def forward(
        self,
        inputs,
        previous_stage_features=None,
    ):
        output = self.dilated_conv(
            inputs
        )

        output = (
            output
            + self.local_attention(
                self.norm_local(output),
                previous_stage_features,
            )
        )

        output = (
            output
            + self.long_term_attention(
                self.norm_long_term(output),
                previous_stage_features,
            )
        )

        output = self.output_projection(
            output
        )
        output = self.dropout(output)

        return output + inputs


class LTContextStage(nn.Module):
    def __init__(
        self,
        input_dim,
        model_dim,
        num_classes,
        num_layers,
    ):
        super().__init__()

        self.channel_dropout = nn.Dropout1d(
            CHANNEL_MASKING_PROB
        )
        self.input_projection = nn.Conv1d(
            input_dim,
            model_dim,
            kernel_size=1,
        )

        dilations = [
            min(
                DILATION_FACTOR ** layer_index,
                MAX_EFFECTIVE_DILATION,
            )
            for layer_index in range(
                num_layers
            )
        ]

        self.blocks = nn.ModuleList([
            LTContextBlock(
                model_dim=model_dim,
                dilation=dilation,
                num_heads=NUM_ATTENTION_HEADS,
                local_radius=(
                    LOCAL_ATTENTION_RADIUS
                ),
                long_term_stride=(
                    LONG_TERM_ATTENTION_STRIDE
                ),
                attention_dropout=(
                    ATTENTION_DROPOUT
                ),
                block_dropout=BLOCK_DROPOUT,
            )
            for dilation in dilations
        ])

        self.output_projection = nn.Conv1d(
            model_dim,
            num_classes,
            kernel_size=1,
        )

    def forward(
        self,
        inputs,
        previous_stage_features=None,
    ):
        features = self.input_projection(
            self.channel_dropout(inputs)
        )

        for block in self.blocks:
            features = block(
                features,
                previous_stage_features,
            )

        logits = self.output_projection(
            features
        )

        return logits, features


class LTContextModel(nn.Module):
    """Four-stage LTContext model.

    Stage 1 consumes video features. Refinement stages consume the previous
    class probabilities and use previous-stage temporal features as attention
    values, matching the main official LTContext stage flow.
    """

    def __init__(
        self,
        input_dim,
        num_classes,
    ):
        super().__init__()

        self.stage1 = LTContextStage(
            input_dim=input_dim,
            model_dim=MODEL_DIM,
            num_classes=num_classes,
            num_layers=NUM_LAYERS,
        )

        self.dimension_reduction = (
            nn.Conv1d(
                MODEL_DIM,
                REFINEMENT_DIM,
                kernel_size=1,
            )
        )

        self.refinement_stages = (
            nn.ModuleList([
                LTContextStage(
                    input_dim=num_classes,
                    model_dim=(
                        REFINEMENT_DIM
                    ),
                    num_classes=num_classes,
                    num_layers=NUM_LAYERS,
                )
                for _ in range(
                    NUM_STAGES - 1
                )
            ])
        )

    def forward(self, inputs):
        logits, features = self.stage1(
            inputs
        )
        outputs = [logits]

        previous_features = (
            self.dimension_reduction(
                features
            )
        )

        for stage in self.refinement_stages:
            logits, features = stage(
                F.softmax(
                    logits,
                    dim=1,
                ),
                previous_stage_features=(
                    previous_features
                ),
            )

            outputs.append(logits)
            previous_features = features

        return torch.stack(
            outputs,
            dim=0,
        )


def create_model():
    return LTContextModel(
        input_dim=EXPECTED_FEATURE_DIM,
        num_classes=num_classes,
    ).to(DEVICE)


test_model = create_model()

num_parameters = sum(
    parameter.numel()
    for parameter in test_model.parameters()
)

print("Parameters:", num_parameters)

with torch.no_grad():
    test_input = torch.zeros(
        2,
        EXPECTED_FEATURE_DIM,
        EXPECTED_TEMPORAL_LENGTH,
        device=DEVICE,
    )
    test_output = test_model(
        test_input
    )

print(
    "Test output shape:",
    tuple(test_output.shape),
)

assert test_output.shape == (
    NUM_STAGES,
    2,
    num_classes,
    EXPECTED_TEMPORAL_LENGTH,
)

del test_model, test_input, test_output
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Parameters: 881832
Test output shape: (4, 2, 202, 16)


## 12. TAS evaluation metrics

In [12]:
CANDIDATE_IGNORE_CLASSES = {
    "background",
    "SIL",
    "silence",
}
ignore_class_ids = {
    label_to_id[label]
    for label in CANDIDATE_IGNORE_CLASSES
    if label in label_to_id
}

print("Ignored class IDs:", ignore_class_ids)


def collapse_segments(
    frame_labels,
    ignored_ids=None,
):
    ignored_ids = set(
        ignored_ids or []
    )

    labels = []
    starts = []
    ends = []

    active_label = None

    for index, label in enumerate(
        frame_labels
    ):
        label = int(label)

        if label in ignored_ids:
            if active_label is not None:
                ends.append(index)
                active_label = None
            continue

        if label != active_label:
            if active_label is not None:
                ends.append(index)

            labels.append(label)
            starts.append(index)
            active_label = label

    if active_label is not None:
        ends.append(
            len(frame_labels)
        )

    return labels, starts, ends


def levenshtein_distance(
    predicted,
    target,
):
    rows = len(predicted) + 1
    columns = len(target) + 1

    distance = np.zeros(
        (rows, columns),
        dtype=np.int32,
    )

    distance[:, 0] = np.arange(rows)
    distance[0, :] = np.arange(columns)

    for row in range(1, rows):
        for column in range(
            1,
            columns,
        ):
            substitution_cost = (
                0
                if predicted[row - 1]
                == target[column - 1]
                else 1
            )

            distance[row, column] = min(
                distance[
                    row - 1,
                    column,
                ] + 1,
                distance[
                    row,
                    column - 1,
                ] + 1,
                distance[
                    row - 1,
                    column - 1,
                ] + substitution_cost,
            )

    return int(
        distance[-1, -1]
    )


def edit_score_single(
    prediction,
    target,
    ignored_ids=None,
):
    predicted_segments, _, _ = (
        collapse_segments(
            prediction,
            ignored_ids,
        )
    )
    target_segments, _, _ = (
        collapse_segments(
            target,
            ignored_ids,
        )
    )

    if (
        not predicted_segments
        and not target_segments
    ):
        return 100.0

    denominator = max(
        len(predicted_segments),
        len(target_segments),
    )

    if denominator == 0:
        return 0.0

    distance = levenshtein_distance(
        predicted_segments,
        target_segments,
    )

    return (
        1.0
        - distance / denominator
    ) * 100.0


def f_score_single(
    prediction,
    target,
    overlap,
    ignored_ids=None,
):
    (
        predicted_labels,
        predicted_starts,
        predicted_ends,
    ) = collapse_segments(
        prediction,
        ignored_ids,
    )

    (
        target_labels,
        target_starts,
        target_ends,
    ) = collapse_segments(
        target,
        ignored_ids,
    )

    true_positives = 0
    false_positives = 0

    hits = np.zeros(
        len(target_labels),
        dtype=np.float32,
    )

    for predicted_index in range(
        len(predicted_labels)
    ):
        best_iou = 0.0
        best_target_index = -1

        for target_index in range(
            len(target_labels)
        ):
            if (
                predicted_labels[
                    predicted_index
                ]
                != target_labels[
                    target_index
                ]
            ):
                continue

            intersection = (
                min(
                    predicted_ends[
                        predicted_index
                    ],
                    target_ends[
                        target_index
                    ],
                )
                - max(
                    predicted_starts[
                        predicted_index
                    ],
                    target_starts[
                        target_index
                    ],
                )
            )

            union = (
                max(
                    predicted_ends[
                        predicted_index
                    ],
                    target_ends[
                        target_index
                    ],
                )
                - min(
                    predicted_starts[
                        predicted_index
                    ],
                    target_starts[
                        target_index
                    ],
                )
            )

            iou = (
                max(intersection, 0)
                / union
                if union > 0
                else 0.0
            )

            if iou > best_iou:
                best_iou = iou
                best_target_index = (
                    target_index
                )

        if (
            best_iou >= overlap
            and best_target_index >= 0
            and hits[
                best_target_index
            ] == 0
        ):
            true_positives += 1
            hits[
                best_target_index
            ] = 1
        else:
            false_positives += 1

    false_negatives = (
        len(target_labels)
        - int(hits.sum())
    )

    return (
        true_positives,
        false_positives,
        false_negatives,
    )


def compute_metrics(
    predictions,
    targets,
    ignored_ids=None,
):
    total_correct = 0
    total_positions = 0
    edit_scores = []

    f_statistics = {
        0.10: [0, 0, 0],
        0.25: [0, 0, 0],
        0.50: [0, 0, 0],
    }

    for sequence_id, prediction in (
        predictions.items()
    ):
        target = targets[sequence_id]

        if len(prediction) != len(target):
            raise ValueError(
                f"Length mismatch for {sequence_id}"
            )

        total_correct += int(
            (
                prediction == target
            ).sum()
        )
        total_positions += len(target)

        edit_scores.append(
            edit_score_single(
                prediction,
                target,
                ignored_ids,
            )
        )

        for overlap in f_statistics:
            tp, fp, fn = f_score_single(
                prediction,
                target,
                overlap,
                ignored_ids,
            )

            f_statistics[
                overlap
            ][0] += tp
            f_statistics[
                overlap
            ][1] += fp
            f_statistics[
                overlap
            ][2] += fn

    metrics = {
        "acc": (
            100.0
            * total_correct
            / max(total_positions, 1)
        ),
        "edit": (
            float(np.mean(edit_scores))
            if edit_scores
            else 0.0
        ),
    }

    for overlap, (
        tp,
        fp,
        fn,
    ) in f_statistics.items():
        precision = (
            tp / max(tp + fp, 1e-8)
        )
        recall = (
            tp / max(tp + fn, 1e-8)
        )

        f1 = (
            2.0
            * precision
            * recall
            / max(
                precision + recall,
                1e-8,
            )
        )

        metrics[
            f"f1@{int(overlap * 100)}"
        ] = 100.0 * f1

    return metrics


print("Metric implementation ready.")

Ignored class IDs: set()
Metric implementation ready.


## 13. LTContext loss and evaluation helpers

In [13]:
def ltcontext_loss(
    outputs,
    targets,
):
    # outputs: [stages, batch, classes, time]
    # targets: [batch, time]
    total_loss = torch.zeros(
        (),
        device=outputs.device,
    )

    for stage_index in range(
        outputs.shape[0]
    ):
        logits = outputs[
            stage_index
        ]

        cross_entropy = F.cross_entropy(
            logits.permute(
                0,
                2,
                1,
            ).reshape(
                -1,
                logits.shape[1],
            ),
            targets.reshape(-1),
        )

        log_probabilities = F.log_softmax(
            logits,
            dim=1,
        )

        smoothing = F.mse_loss(
            log_probabilities[
                :,
                :,
                1:,
            ],
            log_probabilities.detach()[
                :,
                :,
                :-1,
            ],
            reduction="none",
        )

        smoothing = torch.clamp(
            smoothing,
            min=0.0,
            max=SMOOTHING_CLIP_VALUE,
        ).mean()

        total_loss = (
            total_loss
            + cross_entropy
            + SMOOTHING_LOSS_WEIGHT
            * smoothing
        )

    return total_loss


@torch.no_grad()
def evaluate_model(
    model,
    loader,
    save_predictions=False,
    prediction_dir=None,
):
    model.eval()

    predictions = {}
    targets = {}

    if save_predictions:
        prediction_dir = Path(
            prediction_dir
        )
        prediction_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

    for batch in loader:
        features = batch[
            "features"
        ].to(
            DEVICE,
            non_blocking=True,
        )
        labels = batch[
            "labels"
        ].cpu().numpy()
        sequence_ids = list(
            batch["sequence_id"]
        )

        outputs = model(features)
        final_logits = outputs[-1]

        predicted = (
            final_logits.argmax(
                dim=1
            )
            .detach()
            .cpu()
            .numpy()
        )

        for batch_index, sequence_id in enumerate(
            sequence_ids
        ):
            prediction = predicted[
                batch_index
            ].astype(np.int64)
            target = labels[
                batch_index
            ].astype(np.int64)

            predictions[
                sequence_id
            ] = prediction
            targets[
                sequence_id
            ] = target

            if save_predictions:
                prediction_labels = [
                    id_to_label[int(value)]
                    for value in prediction
                ]

                (
                    prediction_dir
                    / f"{sequence_id}.txt"
                ).write_text(
                    "\n".join(
                        prediction_labels
                    )
                    + "\n",
                    encoding="utf-8",
                )

    metrics = compute_metrics(
        predictions,
        targets,
        ignored_ids=ignore_class_ids,
    )

    return (
        metrics,
        predictions,
        targets,
    )


def rounded_metrics(metrics):
    return {
        key: round(float(value), 2)
        for key, value in metrics.items()
    }


def torch_load_checkpoint(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


print("LTContext loss and evaluation helpers ready.")

LTContext loss and evaluation helpers ready.


## 14. Controlled LTContext training function

In [14]:
def run_visual_only_experiment(
    representation,
    representation_config,
):
    display_name = representation_config[
        "display_name"
    ]
    data_root = Path(
        representation_config[
            "data_root"
        ]
    )

    experiment_root = (
        OUT_ROOT
        / representation
        / RUN_MODE
    )
    model_dir = (
        experiment_root / "models"
    )
    prediction_dir = (
        experiment_root / "predictions"
    )
    result_dir = (
        experiment_root / "results"
    )

    for directory in [
        experiment_root,
        model_dir,
        prediction_dir,
        result_dir,
    ]:
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    final_summary_path = (
        result_dir
        / "final_summary.json"
    )

    if (
        final_summary_path.exists()
        and not FORCE_RETRAIN_COMPLETED
    ):
        previous_summary = json.loads(
            final_summary_path.read_text(
                encoding="utf-8"
            )
        )

        if (
            previous_summary.get(
                "status"
            )
            == "completed"
        ):
            print(
                f"\nSkipping completed experiment: "
                f"{representation}"
            )
            return previous_summary

    feature_dir = (
        data_root / "features"
    )

    if STANDARDIZE_FEATURES:
        train_mean, train_std = (
            compute_train_stats(
                feature_dir,
                train_ids,
            )
        )
    else:
        train_mean = np.zeros(
            EXPECTED_FEATURE_DIM,
            dtype=np.float32,
        )
        train_std = np.ones(
            EXPECTED_FEATURE_DIM,
            dtype=np.float32,
        )

    mean_path = (
        result_dir
        / "train_feature_mean.npy"
    )
    std_path = (
        result_dir
        / "train_feature_std.npy"
    )

    np.save(mean_path, train_mean)
    np.save(std_path, train_std)

    train_dataset = (
        Assembly101FeatureDataset(
            train_ids,
            data_root,
            label_to_id,
            train_mean,
            train_std,
        )
    )
    val_dataset = (
        Assembly101FeatureDataset(
            val_ids,
            data_root,
            label_to_id,
            train_mean,
            train_std,
        )
    )
    test_dataset = (
        Assembly101FeatureDataset(
            test_ids,
            data_root,
            label_to_id,
            train_mean,
            train_std,
        )
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    sample_batch = next(
        iter(train_loader)
    )

    print("\n" + "=" * 80)
    print("EXPERIMENT:", representation)
    print("=" * 80)
    print("Display name:", display_name)
    print(
        "Train/val/test:",
        len(train_dataset),
        len(val_dataset),
        len(test_dataset),
    )
    print(
        "Sample feature batch:",
        tuple(
            sample_batch[
                "features"
            ].shape
        ),
    )
    print(
        "Sample label batch:",
        tuple(
            sample_batch[
                "labels"
            ].shape
        ),
    )
    print(
        "Feature mean range:",
        float(train_mean.min()),
        float(train_mean.max()),
    )
    print(
        "Feature std range:",
        float(train_std.min()),
        float(train_std.max()),
    )

    set_global_seed(SEED)
    model = create_model()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    warmup_epochs = int(
        RUN_CFG["warmup_epochs"]
    )
    total_epochs = int(
        RUN_CFG["epochs"]
    )

    def learning_rate_factor(epoch_index):
        # LambdaLR calls this with zero-based epoch indices.
        current_epoch = epoch_index + 1

        if (
            warmup_epochs > 0
            and current_epoch
            <= warmup_epochs
        ):
            return (
                current_epoch
                / warmup_epochs
            )

        cosine_length = max(
            total_epochs
            - warmup_epochs,
            1,
        )
        cosine_position = min(
            max(
                current_epoch
                - warmup_epochs,
                0,
            ),
            cosine_length,
        )

        return 0.5 * (
            1.0
            + math.cos(
                math.pi
                * cosine_position
                / cosine_length
            )
        )

    scheduler = (
        torch.optim.lr_scheduler
        .LambdaLR(
            optimizer,
            lr_lambda=learning_rate_factor,
        )
    )

    best_model_path = (
        model_dir / "best_model.pt"
    )
    last_checkpoint_path = (
        model_dir
        / "last_checkpoint.pt"
    )
    history_path = (
        result_dir
        / "training_history.csv"
    )

    start_epoch = 1
    best_score = -math.inf
    best_epoch = None
    best_val_metrics = None
    no_improvement_evals = 0
    history = []

    if (
        RESUME_IF_AVAILABLE
        and last_checkpoint_path.exists()
    ):
        checkpoint = torch_load_checkpoint(
            last_checkpoint_path
        )

        if (
            checkpoint.get(
                "representation"
            )
            == representation
            and checkpoint.get(
                "run_mode"
            )
            == RUN_MODE
        ):
            model.load_state_dict(
                checkpoint[
                    "model_state_dict"
                ]
            )
            optimizer.load_state_dict(
                checkpoint[
                    "optimizer_state_dict"
                ]
            )
            scheduler.load_state_dict(
                checkpoint[
                    "scheduler_state_dict"
                ]
            )

            start_epoch = (
                int(checkpoint["epoch"])
                + 1
            )
            best_score = float(
                checkpoint.get(
                    "best_score",
                    -math.inf,
                )
            )
            best_epoch = checkpoint.get(
                "best_epoch"
            )
            best_val_metrics = (
                checkpoint.get(
                    "best_val_metrics"
                )
            )
            no_improvement_evals = int(
                checkpoint.get(
                    "no_improvement_evals",
                    0,
                )
            )
            history = list(
                checkpoint.get(
                    "history",
                    [],
                )
            )

            print(
                "Resuming from epoch:",
                start_epoch,
            )

    training_started = time.time()
    stopped_early = False

    for epoch in range(
        start_epoch,
        RUN_CFG["epochs"] + 1,
    ):
        model.train()
        epoch_losses = []

        for batch in train_loader:
            features = batch[
                "features"
            ].to(
                DEVICE,
                non_blocking=True,
            )
            labels = batch[
                "labels"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            outputs = model(features)
            loss = ltcontext_loss(
                outputs,
                labels,
            )

            loss.backward()

            if GRAD_CLIP is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP,
                )

            optimizer.step()

            epoch_losses.append(
                float(
                    loss.detach()
                    .cpu()
                    .item()
                )
            )

        scheduler.step()

        history_row = {
            "epoch": epoch,
            "train_loss": float(
                np.mean(epoch_losses)
            ),
            "learning_rate": float(
                scheduler.get_last_lr()[0]
            ),
        }

        evaluate_now = (
            epoch == 1
            or epoch
            % RUN_CFG["eval_every"]
            == 0
            or epoch
            == RUN_CFG["epochs"]
        )

        if evaluate_now:
            (
                val_metrics,
                _,
                _,
            ) = evaluate_model(
                model,
                val_loader,
                save_predictions=False,
            )

            history_row.update({
                f"val_{key}": value
                for key, value in (
                    val_metrics.items()
                )
            })

            # Model selection uses validation data only.
            selection_score = (
                val_metrics["f1@25"]
                + 0.01
                * val_metrics["edit"]
            )

            improved = (
                selection_score
                > best_score
            )

            if improved:
                best_score = (
                    selection_score
                )
                best_epoch = epoch
                best_val_metrics = dict(
                    val_metrics
                )
                no_improvement_evals = 0

                torch.save(
                    {
                        "epoch": epoch,
                        "representation": (
                            representation
                        ),
                        "run_mode": RUN_MODE,
                        "model_state_dict": (
                            model.state_dict()
                        ),
                        "validation_metrics": (
                            val_metrics
                        ),
                        "selection_score": (
                            selection_score
                        ),
                        "model_config": {
                            "num_stages": NUM_STAGES,
                            "num_layers": NUM_LAYERS,
                            "model_dim": MODEL_DIM,
                            "refinement_dim": REFINEMENT_DIM,
                            "num_attention_heads": NUM_ATTENTION_HEADS,
                            "local_attention_radius": LOCAL_ATTENTION_RADIUS,
                            "long_term_attention_stride": LONG_TERM_ATTENTION_STRIDE,
                            "block_dropout": BLOCK_DROPOUT,
                            "input_dim": (
                                EXPECTED_FEATURE_DIM
                            ),
                            "num_classes": (
                                num_classes
                            ),
                        },
                    },
                    best_model_path,
                )
            else:
                no_improvement_evals += 1

            print(
                f"{representation} "
                f"epoch {epoch:03d} "
                f"loss={history_row['train_loss']:.4f} "
                f"val_acc={val_metrics['acc']:.2f} "
                f"val_edit={val_metrics['edit']:.2f} "
                f"val_f1@10={val_metrics['f1@10']:.2f} "
                f"val_f1@25={val_metrics['f1@25']:.2f} "
                f"val_f1@50={val_metrics['f1@50']:.2f} "
                f"best={'yes' if improved else 'no'}"
            )
        else:
            print(
                f"{representation} "
                f"epoch {epoch:03d} "
                f"loss={history_row['train_loss']:.4f}"
            )

        history.append(history_row)

        torch.save(
            {
                "epoch": epoch,
                "representation": (
                    representation
                ),
                "run_mode": RUN_MODE,
                "model_state_dict": (
                    model.state_dict()
                ),
                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),
                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),
                "best_score": (
                    best_score
                ),
                "best_epoch": (
                    best_epoch
                ),
                "best_val_metrics": (
                    best_val_metrics
                ),
                "no_improvement_evals": (
                    no_improvement_evals
                ),
                "history": history,
            },
            last_checkpoint_path,
        )

        pd.DataFrame(
            history
        ).to_csv(
            history_path,
            index=False,
        )

        patience = RUN_CFG[
            "patience_evals"
        ]

        if (
            evaluate_now
            and patience is not None
            and no_improvement_evals
            >= patience
        ):
            stopped_early = True
            print(
                f"Early stopping after "
                f"{no_improvement_evals} "
                f"validation evaluations "
                f"without improvement."
            )
            break

    training_minutes = (
        time.time()
        - training_started
    ) / 60.0

    if not best_model_path.exists():
        raise RuntimeError(
            f"No best checkpoint was saved "
            f"for {representation}"
        )

    best_checkpoint = (
        torch_load_checkpoint(
            best_model_path
        )
    )
    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )
    model.eval()

    final_val_metrics, _, _ = (
        evaluate_model(
            model,
            val_loader,
            save_predictions=False,
        )
    )

    representation_prediction_dir = (
        prediction_dir
        / "best_checkpoint_test"
    )

    (
        test_metrics,
        test_predictions,
        test_targets,
    ) = evaluate_model(
        model,
        test_loader,
        save_predictions=True,
        prediction_dir=(
            representation_prediction_dir
        ),
    )

    if (
        len(
            list(
                representation_prediction_dir.glob(
                    "*.txt"
                )
            )
        )
        != len(test_ids)
    ):
        raise RuntimeError(
            "Unexpected number of test "
            "prediction files."
        )

    final_metrics_row = {
        "experiment": (
            f"assembly101_{representation}_"
            f"visual_only_mstcn"
        ),
        "representation": representation,
        "features": (
            representation_config[
                "feature_description"
            ]
        ),
        "feature_shape": (
            f"[{EXPECTED_FEATURE_DIM}, "
            f"{EXPECTED_TEMPORAL_LENGTH}]"
        ),
        "model": (
            "LTContext adapted MultiStageModel"
        ),
        "training_input": "video only",
        "inference_input": "video only",
        "split_protocol": (
            "official train/val/test; "
            "checkpoint selected on validation only"
        ),
        "best_epoch": int(
            best_checkpoint["epoch"]
        ),
        "val_acc": round(
            final_val_metrics["acc"],
            2,
        ),
        "val_edit": round(
            final_val_metrics["edit"],
            2,
        ),
        "val_f1@10": round(
            final_val_metrics["f1@10"],
            2,
        ),
        "val_f1@25": round(
            final_val_metrics["f1@25"],
            2,
        ),
        "val_f1@50": round(
            final_val_metrics["f1@50"],
            2,
        ),
        "test_acc": round(
            test_metrics["acc"],
            2,
        ),
        "test_edit": round(
            test_metrics["edit"],
            2,
        ),
        "test_f1@10": round(
            test_metrics["f1@10"],
            2,
        ),
        "test_f1@25": round(
            test_metrics["f1@25"],
            2,
        ),
        "test_f1@50": round(
            test_metrics["f1@50"],
            2,
        ),
    }

    final_metrics_df = pd.DataFrame([
        final_metrics_row
    ])
    final_metrics_path = (
        result_dir
        / "final_metrics.csv"
    )
    final_metrics_md_path = (
        result_dir
        / "final_metrics.md"
    )

    final_metrics_df.to_csv(
        final_metrics_path,
        index=False,
    )
    final_metrics_md_path.write_text(
        final_metrics_df.to_markdown(
            index=False
        ),
        encoding="utf-8",
    )

    summary = {
        "status": "completed",
        "run_mode": RUN_MODE,
        "experiment": (
            final_metrics_row[
                "experiment"
            ]
        ),
        "representation": (
            representation
        ),
        "input_dataset_root": str(
            data_root
        ),
        "output_root": str(
            experiment_root
        ),
        "feature_shape": [
            EXPECTED_FEATURE_DIM,
            EXPECTED_TEMPORAL_LENGTH,
        ],
        "num_classes": num_classes,
        "num_train_sequences": len(
            train_ids
        ),
        "num_val_sequences": len(
            val_ids
        ),
        "num_test_sequences": len(
            test_ids
        ),
        "selection_protocol": (
            "best checkpoint selected by "
            "validation F1@25 + 0.01 * validation Edit; "
            "test evaluated once after selection"
        ),
        "model": {
            "type": (
                "LTContext adapted MultiStageModel"
            ),
            "num_stages": NUM_STAGES,
            "num_layers": NUM_LAYERS,
            "model_dim": MODEL_DIM,
            "refinement_dim": REFINEMENT_DIM,
            "num_attention_heads": NUM_ATTENTION_HEADS,
            "local_attention_radius": LOCAL_ATTENTION_RADIUS,
            "long_term_attention_stride": LONG_TERM_ATTENTION_STRIDE,
            "block_dropout": BLOCK_DROPOUT,
            "num_parameters": (
                sum(
                    parameter.numel()
                    for parameter
                    in model.parameters()
                )
            ),
        },
        "training": {
            "maximum_epochs": (
                RUN_CFG["epochs"]
            ),
            "best_epoch": int(
                best_checkpoint["epoch"]
            ),
            "stopped_early": (
                stopped_early
            ),
            "training_minutes_this_run": (
                training_minutes
            ),
            "batch_size": BATCH_SIZE,
            "learning_rate": (
                LEARNING_RATE
            ),
            "weight_decay": (
                WEIGHT_DECAY
            ),
            "smoothing_loss_weight": (
                SMOOTHING_LOSS_WEIGHT
            ),
            "warmup_epochs": RUN_CFG["warmup_epochs"],
            "standardize_features": (
                STANDARDIZE_FEATURES
            ),
            "seed": SEED,
        },
        "validation_metrics": (
            rounded_metrics(
                final_val_metrics
            )
        ),
        "test_metrics": (
            rounded_metrics(
                test_metrics
            )
        ),
        "paths": {
            "best_model": str(
                best_model_path
            ),
            "last_checkpoint": str(
                last_checkpoint_path
            ),
            "training_history": str(
                history_path
            ),
            "final_metrics_csv": str(
                final_metrics_path
            ),
            "test_predictions": str(
                representation_prediction_dir
            ),
            "train_feature_mean": str(
                mean_path
            ),
            "train_feature_std": str(
                std_path
            ),
        },
        "note": (
            "Visual-only LTContext baseline. "
            "No text embeddings are used. "
            "The attention configuration is adapted to T=16."
        ),
    }

    final_summary_path.write_text(
        json.dumps(
            summary,
            indent=2,
        ),
        encoding="utf-8",
    )

    display(final_metrics_df)
    print(
        json.dumps(
            summary,
            indent=2,
        )
    )

    del (
        model,
        optimizer,
        scheduler,
        train_loader,
        val_loader,
        test_loader,
        train_dataset,
        val_dataset,
        test_dataset,
        sample_batch,
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

## 15. Train both visual-only LTContext baselines

In [15]:
experiment_summaries = {}

for representation, representation_config in (
    REPRESENTATIONS.items()
):
    experiment_summaries[
        representation
    ] = run_visual_only_experiment(
        representation,
        representation_config,
    )

print("\nCompleted representations:")
for representation, summary in (
    experiment_summaries.items()
):
    print(
        representation,
        "->",
        summary["status"],
        summary["test_metrics"],
    )

Train feature statistics:   0%|          | 0/393 [00:00<?, ?it/s]


EXPERIMENT: procedurevrl_hidden
Display name: ProcedureVRL hidden
Train/val/test: 393 120 167
Sample feature batch: (64, 512, 16)
Sample label batch: (64, 16)
Feature mean range: -2.022263288497925 4.081971168518066
Feature std range: 0.1453845053911209 0.5361588001251221
procedurevrl_hidden epoch 001 loss=22.7205 val_acc=0.42 val_edit=0.82 val_f1@10=0.42 val_f1@25=0.42 val_f1@50=0.11 best=yes
procedurevrl_hidden epoch 002 loss=22.5123
procedurevrl_hidden epoch 003 loss=22.3137
procedurevrl_hidden epoch 004 loss=22.0767
procedurevrl_hidden epoch 005 loss=21.8728 val_acc=10.21 val_edit=6.11 val_f1@10=8.03 val_f1@25=6.14 val_f1@50=1.89 best=yes
procedurevrl_hidden epoch 006 loss=21.6529
procedurevrl_hidden epoch 007 loss=21.4197
procedurevrl_hidden epoch 008 loss=21.1748
procedurevrl_hidden epoch 009 loss=21.0088
procedurevrl_hidden epoch 010 loss=20.6673 val_acc=15.52 val_edit=8.21 val_f1@10=11.35 val_f1@25=8.37 val_f1@50=2.70 best=yes
procedurevrl_hidden epoch 011 loss=20.3456
procedu

,experiment,representation,features,feature_shape,model,training_input,inference_input,split_protocol,best_epoch,val_acc,val_edit,val_f1@10,val_f1@25,val_f1@50,test_acc,test_edit,test_f1@10,test_f1@25,test_f1@50
0,assembly101_procedurevrl_hidden_visual_only_mstcn,procedurevrl_hidden,ProcedureVRL model.head hidden video embeddings,"[512, 16]",LTContext adapted MultiStageModel,video only,video only,official train/val/test; checkpoint selected on validation only,80,29.64,22.58,26.35,23.01,14.98,27.1,22.8,25.11,20.05,11.92


{
  "status": "completed",
  "run_mode": "full",
  "experiment": "assembly101_procedurevrl_hidden_visual_only_mstcn",
  "representation": "procedurevrl_hidden",
  "input_dataset_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/procedurevrl_hidden",
  "output_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/procedurevrl_hidden/full",
  "feature_shape": [
    512,
    16
  ],
  "num_classes": 202,
  "num_train_sequences": 393,
  "num_val_sequences": 120,
  "num_test_sequences": 167,
  "selection_protocol": "best checkpoint selected by validation F1@25 + 0.01 * validation Edit; test evaluated once after selection",
  "model": {
    "type": "LTContext adapted MultiStageModel",
    "num_stages": 4,
    "num_layers": 9,
    "model_dim": 64,
    "refinement_dim": 32,
    "num_attention_heads": 4,
    "local_at

Train feature statistics:   0%|          | 0/393 [00:00<?, ?it/s]


EXPERIMENT: clip_vitb16
Display name: CLIP ViT-B/16
Train/val/test: 393 120 167
Sample feature batch: (64, 512, 16)
Sample label batch: (64, 16)
Feature mean range: -0.3327343165874481 0.6023579835891724
Feature std range: 0.004433516878634691 0.04085450991988182
clip_vitb16 epoch 001 loss=22.7284 val_acc=0.73 val_edit=1.74 val_f1@10=0.95 val_f1@25=0.47 val_f1@50=0.24 best=yes
clip_vitb16 epoch 002 loss=22.5893
clip_vitb16 epoch 003 loss=22.3319
clip_vitb16 epoch 004 loss=22.1390
clip_vitb16 epoch 005 loss=21.9222 val_acc=7.71 val_edit=5.58 val_f1@10=5.50 val_f1@25=4.33 val_f1@50=1.64 best=yes
clip_vitb16 epoch 006 loss=21.6743
clip_vitb16 epoch 007 loss=21.4254
clip_vitb16 epoch 008 loss=21.1519
clip_vitb16 epoch 009 loss=21.0222
clip_vitb16 epoch 010 loss=20.6496 val_acc=12.86 val_edit=8.43 val_f1@10=9.93 val_f1@25=7.91 val_f1@50=2.45 best=yes
clip_vitb16 epoch 011 loss=20.3631
clip_vitb16 epoch 012 loss=20.0172
clip_vitb16 epoch 013 loss=19.6809
clip_vitb16 epoch 014 loss=19.1242
c

,experiment,representation,features,feature_shape,model,training_input,inference_input,split_protocol,best_epoch,val_acc,val_edit,val_f1@10,val_f1@25,val_f1@50,test_acc,test_edit,test_f1@10,test_f1@25,test_f1@50
0,assembly101_clip_vitb16_visual_only_mstcn,clip_vitb16,CLIP ViT-B/16 normalized visual frame embeddings,"[512, 16]",LTContext adapted MultiStageModel,video only,video only,official train/val/test; checkpoint selected on validation only,105,30.78,23.82,26.83,23.53,16.13,27.1,22.7,25.62,21.96,13.39


{
  "status": "completed",
  "run_mode": "full",
  "experiment": "assembly101_clip_vitb16_visual_only_mstcn",
  "representation": "clip_vitb16",
  "input_dataset_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16",
  "output_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/clip_vitb16/full",
  "feature_shape": [
    512,
    16
  ],
  "num_classes": 202,
  "num_train_sequences": 393,
  "num_val_sequences": 120,
  "num_test_sequences": 167,
  "selection_protocol": "best checkpoint selected by validation F1@25 + 0.01 * validation Edit; test evaluated once after selection",
  "model": {
    "type": "LTContext adapted MultiStageModel",
    "num_stages": 4,
    "num_layers": 9,
    "model_dim": 64,
    "refinement_dim": 32,
    "num_attention_heads": 4,
    "local_attention_radius": 2,
    "long_te

## 16. Compare LTContext representations and MS-TCN baselines

In [16]:
ltcontext_comparison_rows = []

for representation, summary in (
    experiment_summaries.items()
):
    row = {
        "temporal_model": "LTContext",
        "representation": representation,
        "feature_shape": "[512, 16]",
        "best_epoch": (
            summary["training"][
                "best_epoch"
            ]
        ),
        "val_acc": (
            summary[
                "validation_metrics"
            ]["acc"]
        ),
        "val_edit": (
            summary[
                "validation_metrics"
            ]["edit"]
        ),
        "val_f1@10": (
            summary[
                "validation_metrics"
            ]["f1@10"]
        ),
        "val_f1@25": (
            summary[
                "validation_metrics"
            ]["f1@25"]
        ),
        "val_f1@50": (
            summary[
                "validation_metrics"
            ]["f1@50"]
        ),
        "test_acc": (
            summary[
                "test_metrics"
            ]["acc"]
        ),
        "test_edit": (
            summary[
                "test_metrics"
            ]["edit"]
        ),
        "test_f1@10": (
            summary[
                "test_metrics"
            ]["f1@10"]
        ),
        "test_f1@25": (
            summary[
                "test_metrics"
            ]["f1@25"]
        ),
        "test_f1@50": (
            summary[
                "test_metrics"
            ]["f1@50"]
        ),
    }
    ltcontext_comparison_rows.append(
        row
    )

ltcontext_comparison_df = (
    pd.DataFrame(
        ltcontext_comparison_rows
    )
    .sort_values(
        [
            "test_f1@25",
            "test_edit",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

ltcontext_comparison_path = (
    OUT_ROOT
    / (
        "visual_only_ltcontext_"
        f"comparison_{RUN_MODE}.csv"
    )
)
ltcontext_comparison_md_path = (
    OUT_ROOT
    / (
        "visual_only_ltcontext_"
        f"comparison_{RUN_MODE}.md"
    )
)

ltcontext_comparison_df.to_csv(
    ltcontext_comparison_path,
    index=False,
)
ltcontext_comparison_md_path.write_text(
    ltcontext_comparison_df.to_markdown(
        index=False
    ),
    encoding="utf-8",
)

print("LTContext comparison:")
display(ltcontext_comparison_df)

# --------------------------------------------------------------
# Load notebook 16 MS-TCN results when available.
# --------------------------------------------------------------
mstcn_comparison_path = (
    MS_TCN_RESULTS_ROOT
    / (
        "visual_only_mstcn_"
        f"comparison_{RUN_MODE}.csv"
    )
)

combined_comparison_df = (
    ltcontext_comparison_df.copy()
)

if mstcn_comparison_path.exists():
    mstcn_df = pd.read_csv(
        mstcn_comparison_path
    )

    mstcn_df.insert(
        0,
        "temporal_model",
        "MS-TCN",
    )

    common_columns = [
        column
        for column
        in combined_comparison_df.columns
        if column in mstcn_df.columns
    ]

    combined_comparison_df = pd.concat(
        [
            mstcn_df[
                common_columns
            ],
            combined_comparison_df[
                common_columns
            ],
        ],
        ignore_index=True,
    ).sort_values(
        [
            "test_f1@25",
            "test_edit",
        ],
        ascending=False,
    ).reset_index(drop=True)

    print(
        "Combined MS-TCN + LTContext comparison:"
    )
    display(combined_comparison_df)
else:
    print(
        "MS-TCN comparison was not found:",
        mstcn_comparison_path,
    )

combined_comparison_path = (
    OUT_ROOT
    / (
        "mstcn_vs_ltcontext_"
        f"comparison_{RUN_MODE}.csv"
    )
)
combined_comparison_md_path = (
    OUT_ROOT
    / (
        "mstcn_vs_ltcontext_"
        f"comparison_{RUN_MODE}.md"
    )
)

combined_comparison_df.to_csv(
    combined_comparison_path,
    index=False,
)
combined_comparison_md_path.write_text(
    combined_comparison_df.to_markdown(
        index=False
    ),
    encoding="utf-8",
)

winner_row = (
    combined_comparison_df.iloc[0]
)

final_summary = {
    "status": "completed",
    "run_mode": RUN_MODE,
    "task": (
        "Assembly101 visual-only "
        "LTContext baselines"
    ),
    "implementation": {
        "type": (
            "self-contained LTContext adaptation"
        ),
        "source_architecture": (
            "official LTContext: dilated convolution, "
            "windowed attention, sparse long-term attention, "
            "multi-stage refinement"
        ),
        "adaptation_reason": (
            "The extracted representations contain only "
            "16 temporal positions."
        ),
        "local_attention_radius": (
            LOCAL_ATTENTION_RADIUS
        ),
        "long_term_attention_stride": (
            LONG_TERM_ATTENTION_STRIDE
        ),
        "maximum_effective_dilation": (
            MAX_EFFECTIVE_DILATION
        ),
    },
    "feature_extraction_verified": {
        "recordings": "350/350",
        "sequences": "680/680",
    },
    "protocol": (
        "same official train/val/test split "
        "and identical LTContext training configuration "
        "for ProcedureVRL and CLIP"
    ),
    "checkpoint_selection": (
        "validation only"
    ),
    "representations": (
        experiment_summaries
    ),
    "overall_winner_by_test_f1_at_25": {
        "temporal_model": str(
            winner_row[
                "temporal_model"
            ]
        ),
        "representation": str(
            winner_row[
                "representation"
            ]
        ),
        "test_f1@25": float(
            winner_row[
                "test_f1@25"
            ]
        ),
    },
    "paths": {
        "ltcontext_comparison_csv": str(
            ltcontext_comparison_path
        ),
        "ltcontext_comparison_markdown": str(
            ltcontext_comparison_md_path
        ),
        "combined_comparison_csv": str(
            combined_comparison_path
        ),
        "combined_comparison_markdown": str(
            combined_comparison_md_path
        ),
        "output_root": str(
            OUT_ROOT
        ),
    },
    "next_step": (
        "Train text-assisted concatenation and teacher "
        "baselines using CLIP action-text embeddings."
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}

final_summary_path = (
    OUT_ROOT
    / f"final_summary_{RUN_MODE}.json"
)
final_summary_path.write_text(
    json.dumps(
        final_summary,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Overall winner by test F1@25:",
    final_summary[
        "overall_winner_by_test_f1_at_25"
    ],
)
print("Saved:", ltcontext_comparison_path)
print("Saved:", combined_comparison_path)
print("Saved:", final_summary_path)
print("\nNext notebook:")
print(
    "18_assembly101_text_assisted_"
    "concat_teacher_baselines_COLAB.ipynb"
)

LTContext comparison:


,temporal_model,representation,feature_shape,best_epoch,val_acc,val_edit,val_f1@10,val_f1@25,val_f1@50,test_acc,test_edit,test_f1@10,test_f1@25,test_f1@50
0,LTContext,clip_vitb16,"[512, 16]",105,30.78,23.82,26.83,23.53,16.13,27.1,22.7,25.62,21.96,13.39
1,LTContext,procedurevrl_hidden,"[512, 16]",80,29.64,22.58,26.35,23.01,14.98,27.1,22.8,25.11,20.05,11.92


Combined MS-TCN + LTContext comparison:


,temporal_model,representation,feature_shape,best_epoch,val_acc,val_edit,val_f1@10,val_f1@25,val_f1@50,test_acc,test_edit,test_f1@10,test_f1@25,test_f1@50
0,MS-TCN,clip_vitb16,"[512, 16]",110,29.95,26.01,29.23,25.90,18.97,25.86,25.01,28.00,23.82,17.58
1,MS-TCN,procedurevrl_hidden,"[512, 16]",70,30.26,24.17,28.88,25.15,18.70,26.16,24.74,28.36,23.79,15.72
2,LTContext,clip_vitb16,"[512, 16]",105,30.78,23.82,26.83,23.53,16.13,27.10,22.70,25.62,21.96,13.39
3,LTContext,procedurevrl_hidden,"[512, 16]",80,29.64,22.58,26.35,23.01,14.98,27.10,22.80,25.11,20.05,11.92


Overall winner by test F1@25: {'temporal_model': 'MS-TCN', 'representation': 'clip_vitb16', 'test_f1@25': 23.82}
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/visual_only_ltcontext_comparison_full.csv
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/mstcn_vs_ltcontext_comparison_full.csv
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/final_summary_full.json

Next notebook:
18_assembly101_text_assisted_concat_teacher_baselines_COLAB.ipynb


## Completion criterion

A successful full run ends with:

```text
ProcedureVRL hidden -> completed
CLIP ViT-B/16      -> completed
```

and writes:

```text
.../streaming_visual_features_v1/runs/17_visual_only_ltcontext/
├── procedurevrl_hidden/full/
│   ├── models/
│   ├── predictions/
│   └── results/
├── clip_vitb16/full/
│   ├── models/
│   ├── predictions/
│   └── results/
├── visual_only_ltcontext_comparison_full.csv
├── visual_only_ltcontext_comparison_full.md
├── mstcn_vs_ltcontext_comparison_full.csv
├── mstcn_vs_ltcontext_comparison_full.md
└── final_summary_full.json
```

The next stage is the first text-assisted Assembly101 experiment using the
stored CLIP embeddings for all 202 action classes.